# LAB 3 - Bài 4: Phân lớp Iris bằng Neural Network (10-fold cross validation)
Hồng Kim Dung - 6451071009

## Câu a - Đọc bộ dữ liệu Iris

In [1]:
print("Hồng Kim Dung - 6451071009")
from sklearn.datasets import load_iris

iris = load_iris()
print("Đã đọc bộ dữ liệu Iris:", iris.data.shape, iris.target.shape)
print("Các lớp:", [str(t) for t in iris.target_names])

Hồng Kim Dung - 6451071009
Đã đọc bộ dữ liệu Iris: (150, 4) (150,)
Các lớp: ['setosa', 'versicolor', 'virginica']


## Câu b - Chuẩn hóa dữ liệu về [0,1]

In [2]:
print("Hồng Kim Dung - 6451071009")
import numpy as np

data_max = iris.data.max(axis=0)       # giữ lại để chuẩn hóa mẫu mới ở câu g
X = iris.data / data_max
Y = iris.target
print("Min:", X.min(axis=0))
print("Max:", X.max(axis=0))

Hồng Kim Dung - 6451071009
Min: [0.5443038  0.45454545 0.14492754 0.04      ]
Max: [1. 1. 1. 1.]


## Câu c - Hàm create_model (giống câu 1.h)

In [3]:
print("Hồng Kim Dung - 6451071009")
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input

def create_model():
    model = Sequential([
        Input(shape=(X.shape[1],)),
        Dense(10, activation="relu"),
        Dense(20, activation="relu"),
        Dense(3, activation="softmax"),
    ])
    model.compile(optimizer="adam",
                  loss="sparse_categorical_crossentropy",
                  metrics=["accuracy"])
    return model

create_model().summary()

Hồng Kim Dung - 6451071009


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 10)             │            50 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 20)             │           220 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 3)              │            63 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 333 (1.30 KB)

 Trainable params: 333 (1.30 KB)

 Non-trainable params: 0 (0.00 B)

## Câu d - Huấn luyện bằng 10-fold cross validation (chạy khoảng 5-15 phút)

In [4]:
print("Hồng Kim Dung - 6451071009")
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

kfold = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

models = []
acc_list, prec_list, rec_list, f1_list = [], [], [], []
prec_cls, rec_cls, f1_cls = [], [], []      # chỉ số theo từng class của mỗi fold

for fold, (train_index, val_index) in enumerate(kfold.split(X, Y), start=1):
    model = create_model()
    X_train_fold, X_val_fold = X[train_index], X[val_index]
    y_train_fold, y_val_fold = Y[train_index], Y[val_index]

    model.fit(X_train_fold, y_train_fold, epochs=100, batch_size=1, verbose=0)
    models.append(model)

    y_pred = np.argmax(model.predict(X_val_fold, verbose=0), axis=1)

    acc = accuracy_score(y_val_fold, y_pred)
    p, r, f, _ = precision_recall_fscore_support(y_val_fold, y_pred, average="macro", zero_division=0)
    pc, rc, fc, _ = precision_recall_fscore_support(y_val_fold, y_pred, labels=[0, 1, 2], zero_division=0)

    acc_list.append(acc); prec_list.append(p); rec_list.append(r); f1_list.append(f)
    prec_cls.append(pc); rec_cls.append(rc); f1_cls.append(fc)

    print(f"Fold {fold:2d}: accuracy = {acc:.4f} | precision = {p:.4f} | recall = {r:.4f} | f1 = {f:.4f}")

Hồng Kim Dung - 6451071009
Fold  1: accuracy = 1.0000 | precision = 1.0000 | recall = 1.0000 | f1 = 1.0000
Fold  2: accuracy = 1.0000 | precision = 1.0000 | recall = 1.0000 | f1 = 1.0000
Fold  3: accuracy = 0.9333 | precision = 0.9444 | recall = 0.9333 | f1 = 0.9327
Fold  4: accuracy = 1.0000 | precision = 1.0000 | recall = 1.0000 | f1 = 1.0000


Fold  5: accuracy = 0.9333 | precision = 0.9444 | recall = 0.9333 | f1 = 0.9327


Fold  6: accuracy = 0.8667 | precision = 0.9048 | recall = 0.8667 | f1 = 0.8611
Fold  7: accuracy = 1.0000 | precision = 1.0000 | recall = 1.0000 | f1 = 1.0000
Fold  8: accuracy = 1.0000 | precision = 1.0000 | recall = 1.0000 | f1 = 1.0000
Fold  9: accuracy = 0.9333 | precision = 0.9444 | recall = 0.9333 | f1 = 0.9327
Fold 10: accuracy = 1.0000 | precision = 1.0000 | recall = 1.0000 | f1 = 1.0000


## Câu e - Giá trị trung bình của mô hình

In [5]:
print("Hồng Kim Dung - 6451071009")
print("Average Accuracy :", np.mean(acc_list))
print("Average Precision:", np.mean(prec_list))
print("Average Recall   :", np.mean(rec_list))
print("Average F1-score :", np.mean(f1_list))

Hồng Kim Dung - 6451071009
Average Accuracy : 0.9666666666666666
Average Precision: 0.9738095238095239
Average Recall   : 0.9666666666666666
Average F1-score : 0.9659090909090908


## Câu f - Trung bình Precision, Recall, F1-score theo từng class

In [6]:
print("Hồng Kim Dung - 6451071009")
import pandas as pd

bang = pd.DataFrame({
    "Average Precision": np.mean(prec_cls, axis=0),
    "Average Recall":    np.mean(rec_cls, axis=0),
    "Average F1-score":  np.mean(f1_cls, axis=0),
}, index=[str(t) for t in iris.target_names])
print(bang.round(4))

Hồng Kim Dung - 6451071009
            Average Precision  Average Recall  Average F1-score
setosa                 1.0000            1.00            1.0000
versicolor             0.9833            0.92            0.9437
virginica              0.9381            0.98            0.9540


## Câu g - Dự đoán 3 mẫu mới bằng mô hình có F1-score tốt nhất

In [7]:
print("Hồng Kim Dung - 6451071009")
best_index = np.argmax(f1_list)
best_model = models[best_index]
print("Fold tốt nhất:", best_index + 1, "- F1-score =", f1_list[best_index])

X_new = np.array([[6.2, 2.9, 4.3, 1.3],
                  [5.1, 3.5, 1.4, 0.2],
                  [7.3, 2.8, 6.4, 2.1]])
X_new_scaled = X_new / data_max        # chuẩn hóa cùng thang đo với dữ liệu huấn luyện

y_new = np.argmax(best_model.predict(X_new_scaled, verbose=0), axis=1)
for mau, k in zip(X_new, y_new):
    print(mau, "->", str(iris.target_names[k]))

Hồng Kim Dung - 6451071009
Fold tốt nhất: 1 - F1-score = 1.0
[6.2 2.9 4.3 1.3] -> versicolor
[5.1 3.5 1.4 0.2] -> setosa
[7.3 2.8 6.4 2.1] -> virginica
